# grapheme-aware Evaluation Metrics

A **grapheme-aware** evaluation notebook for comparing a hypothesis or model output against reference text using `graphemes_plusplus.metric`.

These metrics compare grapheme++ clusters instead of raw Unicode code points, which is important for Sinhala, Tamil, Arabic, emoji, and other writing systems with multi-codepoint visible characters.

## Setup


In [ ]:
%pip install sacrebleu

In [ ]:
%pip install graphemes_plusplus

In [1]:
# Install dependencies
# pip install sacrebleu graphemes-plusplus

from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path
import sys

project_root = Path.cwd()
for candidate in (project_root, project_root.parent, project_root.parent.parent):
    if (candidate / 'src').exists():
        project_root = candidate
        break

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from graphemes_plusplus.graphemizer import Graphemizer
from graphemes_plusplus.metric import CER, GraphemeCHRF, charbleu


def quiet_cer(hypothesis: str, reference: str) -> float:
    """Call CER while hiding its current debug print output."""
    with redirect_stdout(StringIO()):
        return CER(hypothesis, reference)


ModuleNotFoundError: No module named 'sacrebleu'

---
## Using Library Metrics

**Reference:** expected or gold text  
**Hypothesis:** predicted text from OCR, ASR, translation, transliteration, or normalization

Use `CER(hypothesis, reference)` for error rate, `GraphemeCHRF().sentence_score(hypothesis, [reference])` for chrF, and `charbleu(reference, hypothesis)` for CharBLEU.

In [ ]:
reference = "සිංහල"
hypothesis = "සිහල"

print(f"Reference : {reference}")
print(f"Hypothesis: {hypothesis}")
print()
print(f"CER(hypothesis, reference) = {quiet_cer(hypothesis, reference):.4f}")
print(f"GraphemeCHRF().sentence_score(...).score = {GraphemeCHRF().sentence_score(hypothesis, [reference]).score:.4f}")
print(f"charbleu(reference, hypothesis) = {charbleu(reference, hypothesis):.4f}")


---
## Grapheme Segmentation Check

**Why:** evaluation metrics should count visible grapheme clusters, not Python string length.  
**Returns:** a list of grapheme++ clusters for each input string.

In [ ]:
examples = ["සිංහල", "ක්‍රමය", "ஸ்ரீ", "வணக்கம்"]

for text in examples:
    graphemes = list(Graphemizer(text))
    print(f"{text:<10} Python len={len(text):<2} grapheme++ len={len(graphemes):<2} {graphemes}")


---
## Character Error Rate (CER)

**Operation:** grapheme-aware Levenshtein distance divided by reference grapheme count.  
**Range:** `0.0` is perfect; larger values mean more errors.  
**Best for:** OCR, ASR, spelling correction, and text normalization.

CER is useful when each insertion, deletion, or substitution error should count directly.

In [ ]:
pairs = [
    ("සිංහල", "සිංහල"),
    ("ක්‍රමය", "ක්මය"),
    ("කනව", "කනවා"),
    ("ஸ்ரி", "ஸ்ரீ"),
]

print(f"{'reference':<12} {'hypothesis':<12} {'CER':>8}")
print("-" * 36)
for reference, hypothesis in pairs:
    print(f"{reference:<12} {hypothesis:<12} {quiet_cer(hypothesis, reference):>8.4f}")


---
## Grapheme chrF

**Compares:** overlapping grapheme n-grams between hypothesis and reference.  
**Range:** `0.0` to `100.0`; higher is better.  
**Best for:** translation, generation, and partial-match evaluation.

chrF gives partial credit when strings share many grapheme sequences, even if the full output is not exact.

In [ ]:
chrf = GraphemeCHRF()

pairs = [
    ("வணக்கம்", "வணக்கம்"),
    ("සිංහල", "සිහල"),
    ("ක්‍රමය", "ක්මය"),
    ("Hello வணக்கம் world", "Hello வணக்கம், world!"),
]

print(f"{'reference':<24} {'hypothesis':<24} {'chrF':>8}")
print("-" * 62)
for reference, hypothesis in pairs:
    score = chrf.sentence_score(hypothesis, [reference]).score
    print(f"{reference:<24} {hypothesis:<24} {score:>8.4f}")


---
## Grapheme chrF++

**Adds:** word n-grams on top of grapheme n-grams.  
**Range:** `0.0` to `100.0`; higher is better.  
**Best for:** sentence-level output where word order should also matter.

Use `GraphemeCHRF(word_order=2)` when you want chrF++ behavior.

In [ ]:
chrf = GraphemeCHRF()
chrf_pp = GraphemeCHRF(word_order=2)

pairs = [
    ("සුබ උදෑසනක්", "සුබ උදෑසනක්"),
    ("මෙය නව මාර්ග වේ", "මෙය නව මාර්ගය වේ"),
    ("இன்று வானிலை மிகவும் அழகாக இருக்கிறது", "இருக்கிறது அழகாக மிகவும் வானிலை இன்று"),
]

print(f"{'reference':<42} {'hypothesis':<42} {'chrF':>8} {'chrF++':>8}")
print("-" * 106)
for reference, hypothesis in pairs:
    score = chrf.sentence_score(hypothesis, [reference]).score
    score_pp = chrf_pp.sentence_score(hypothesis, [reference]).score
    print(f"{reference:<42} {hypothesis:<42} {score:>8.4f} {score_pp:>8.4f}")


---
## CharBLEU

**Compares:** grapheme n-gram precision with a BLEU-style geometric mean.  
**Range:** `0.0` to `1.0`; higher is better.  
**Best for:** compact similarity scores where exact grapheme n-gram matches should be rewarded.

`charbleu(reference, hypothesis)` is useful when you want a BLEU-like score without tokenizing into words.

In [ ]:
pairs = [
    ("එවන්න", "එවන්න"),
    ("කරණ", "කරම"),
    ("කරණල", "කරණ"),
    ("abc", "bac"),
    ("ஸ்ரீ", "ரீ"),
]

print(f"{'reference':<12} {'hypothesis':<12} {'CharBLEU':>10}")
print("-" * 38)
for reference, hypothesis in pairs:
    print(f"{reference:<12} {hypothesis:<12} {charbleu(reference, hypothesis):>10.4f}")


---
## Corpus Evaluation

For a model or dataset, evaluate every hypothesis against its reference, then summarize with mean CER, corpus chrF, corpus chrF++, and mean CharBLEU.

In [ ]:
dataset = [
    ("si-perfect", "සිංහල", "සිංහල"),
    ("si-conjunct", "ක්‍රමය", "ක්මය"),
    ("si-substitution", "කනව", "කනවා"),
    ("ta-perfect", "வணக்கம்", "வணக்கம்"),
    ("ta-grapheme", "ஸ்ரி", "ஸ்ரீ"),
]

print(f"{'id':<16} {'CER':>8} {'chrF':>8} {'CharBLEU':>10}")
print("-" * 48)

cer_scores = []
charbleu_scores = []
hypotheses = []
references = []

for item_id, reference, hypothesis in dataset:
    cer = quiet_cer(hypothesis, reference)
    chrf_score = chrf.sentence_score(hypothesis, [reference]).score
    bleu = charbleu(reference, hypothesis)

    cer_scores.append(cer)
    charbleu_scores.append(bleu)
    hypotheses.append(hypothesis)
    references.append(reference)

    print(f"{item_id:<16} {cer:>8.4f} {chrf_score:>8.2f} {bleu:>10.4f}")

print()
print(f"Mean CER      = {sum(cer_scores) / len(cer_scores):.4f}")
print(f"Corpus chrF   = {chrf.corpus_score(hypotheses, [references]).score:.4f}")
print(f"Corpus chrF++ = {chrf_pp.corpus_score(hypotheses, [references]).score:.4f}")
print(f"Mean CharBLEU = {sum(charbleu_scores) / len(charbleu_scores):.4f}")


---
## Choosing the Right Metric

| Metric | Output | Best for |
|--------|--------|----------|
| **CER** | float error rate, lower is better | OCR, ASR, spelling correction, normalization |
| **Grapheme chrF** | `0.0` to `100.0`, higher is better | Partial overlap in generated text |
| **Grapheme chrF++** | `0.0` to `100.0`, higher is better | Generated sentences where word n-grams matter |
| **CharBLEU** | `0.0` to `1.0`, higher is better | BLEU-like grapheme n-gram similarity |

**Key principle:** use CER when you want to count edit errors, and use chrF / chrF++ / CharBLEU when you want a similarity score.